# BPE from scratch — Solution

Reference implementation for `01_bpe_exercise.ipynb`.

## Setup

In [70]:
from collections import Counter
import itertools

## Solution 1 — `pre_tokenize`

In [ ]:
def pre_tokenize(corpus):
    corpus = [tuple(word)+("</w>",) for sentence in corpus for word in sentence.lower().split()]
    count = Counter(corpus)
    
    return count
    

In [69]:
# Sanity check
corpus = ["the cat sat", "the cat ate"]
word_freqs = pre_tokenize(corpus)
for word, freq in word_freqs.items():
    print(word, '→', freq)
# Expected:
#   ('t', 'h', 'e', '</w>') → 2
#   ('c', 'a', 't', '</w>') → 2
#   ('s', 'a', 't', '</w>') → 1
#   ('a', 't', 'e', '</w>') → 1

('t', 'h', 'e', '</w>') → 2
('c', 'a', 't', '</w>') → 2
('s', 'a', 't', '</w>') → 1
('a', 't', 'e', '</w>') → 1


**Why the `</w>` marker?**  it preserves word boundaries. Without it, after enough merges, the tokenizer couldn't tell whether two adjacent character sequences came from one word or two._

## Solution 2 — `get_pair_counts`

In [ ]:
def get_pair_counts(word_freqs: dict[tuple[str, ...], int]) -> dict[tuple[str, str], int]:
    pair_counts = Counter()

    for word,freq in word_freqs.items():
        for pair in zip(word, word[1:]):
            pair_counts[pair] += freq
    return pair_counts

In [82]:
get_pair_counts(word_freqs)

Counter({('a', 't'): 4,
         ('e', '</w>'): 3,
         ('t', '</w>'): 3,
         ('t', 'h'): 2,
         ('h', 'e'): 2,
         ('c', 'a'): 2,
         ('s', 'a'): 1,
         ('t', 'e'): 1})

**Why count weighted by word frequency?** If a word appears 1000 times in the corpus, every internal pair is effectively seen 1000 times. The compact representation keeps this exact while only iterating over unique words._

## Solution 3 — `merge_pair`

In [ ]:
def merge_pair(
    pair: tuple[str, str],
    word_freqs: dict[tuple[str, ...], int],
) -> dict[tuple[str, ...], int]:
    updated_word_frequency = {}

    for word,freq in word_freqs.items():
        
        i = 0 
        new_word = []
        while i < len(word):
            if i < len(word)-1 and (word[i],word[i+1]) == pair:
                new_word.append(pair[0] + pair[1])
                i += 2
            else:
                new_word.append(word[i])
                i +=1

        updated_word_frequency[tuple(new_word)] = freq

    return updated_word_frequency

In [119]:
merged = merge_pair(('a', 't'), word_freqs)

In [120]:
merged

{('t', 'h', 'e', '</w>'): 2,
 ('c', 'at', '</w>'): 2,
 ('s', 'at', '</w>'): 1,
 ('at', 'e', '</w>'): 1}

## Solution 4 — `BPETokenizer.train`

In [ ]:
class BPETokenizer:
    def __init__(self):
        self.merges = []
        self.vocab = set()

    def train(self, corpus, vocab_size):
        word_freqs = pre_tokenize(corpus)
        self.vocab = set([char for char_list in word_freqs.keys() for char in char_list])
        
        while len(self.vocab) <= vocab_size:
            ## Count Pairs
            pair_counts = get_pair_counts(word_freqs)
            if not pair_counts: break

            ## Add the best pair to the merge to be used at encoding time
            best_pair = max(pair_counts, key=pair_counts.get)
            self.merges.append(best_pair)

            ## Reset word frequency
            word_freqs = merge_pair(best_pair, word_freqs)

            self.vocab.add(best_pair[0] + best_pair[1])

    def _apply_merge(self,tokens,pair):

        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens)-1 and (tokens[i],tokens[i+1]) == pair:
                new_tokens.append(tokens[i]+tokens[i+1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i+=1

        return new_tokens
    

    def encode(self, text):
        final_tokens = []

        for word in text.lower().split():
            tokens = tuple(word) + ("</w>",)
            for pair in self.merges:
                tokens = self._apply_merge(tokens,pair)

            final_tokens.extend(tokens)

        return final_tokens

In [136]:
# Sanity check — train on a tiny corpus
corpus = ["the cat sat", "the cat ate", "the bat sat"] * 5
tok = BPETokenizer()
tok.train(corpus, vocab_size=20)
print('vocab size:', len(tok.vocab))
print('vocab:', sorted(tok.vocab))
print('first few merges:', tok.merges[:5])
# You should see merges that build up common substrings like 't','h' → 'th', then 'th','e' → 'the', etc.

vocab size: 17
vocab: ['</w>', 'a', 'at', 'at</w>', 'ate</w>', 'b', 'bat</w>', 'c', 'cat</w>', 'e', 'e</w>', 'h', 's', 'sat</w>', 't', 'th', 'the</w>']
first few merges: [('a', 't'), ('at', '</w>'), ('e', '</w>'), ('t', 'h'), ('th', 'e</w>')]


**Why store merges as an ordered list?** BPE encoding is deterministic — given the same merge order, the same input produces the same output. The order encodes which merges happened first (the most frequent pairs), which determines how new text gets segmented._

**Edge cases worth thinking about:**
- What if the corpus is empty?
- What if all pairs have the same count (tie-breaking)?
- What happens at vocab_size smaller than the number of unique characters?

## Solution 5 — `encode`

The encode method lives inside `BPETokenizer` (above). The trick: replay merges *in order*, and for each merge scan the word for the pair to apply it. The order matters because later merges depend on earlier ones existing in the vocabulary.

In [137]:
# Sanity check encoding
tokens = tok.encode("the cat")
print(tokens)
# After enough merges, you should see something like ['the</w>', 'cat</w>'] or close.
# Without merges, you'd see individual chars: ['t','h','e','</w>','c','a','t','</w>']

['the</w>', 'cat</w>']


## Run the tests

In [138]:
from tests import run_bpe_tests
run_bpe_tests(BPETokenizer, pre_tokenize, get_pair_counts, merge_pair)

Running BPE...
  ✓ pre_tokenize: basic word frequencies
  ✓ pre_tokenize: end-of-word marker present
  ✓ get_pair_counts: weighted by word frequency
  ✓ merge_pair: replaces pairs in matching words
  ✓ merge_pair: leaves non-matching words alone
  ✓ train: vocab size respects target
  ✓ train: includes all single characters
  ✓ train: produces at least one merge
  ✓ encode: returns list of strings
  ✓ encode: compresses below char count

  All 10 checks passed ✓



True